# 1. Project Title
Automatic Number Plate Recognition (ANPR) System using YOLOv8




# 2. Problem Statement
Manual tracking and recording of vehicle license plates at gates, parking lots, and security checkpoints are time-consuming and prone to human error. The goal of this project is to automate vehicle identification and license plate detection efficiently.


# 3. Why Computer Vision?
Object detection (specifically YOLOv8) is the ideal solution because it can accurately locate and bound license plates in complex street environments, varying lighting conditions, and dynamic angles where traditional image processing fails.


# 4. Dataset
* **Dataset Name:** License Plate Detection Dataset
* **Source/Link:** Kaggle Dataset Repository
* **Number of images:** 1,694 images total 
  * Training set: 1,526 images
  * Validation set: 84 images
  * Test set: 84 images
* **Classes:** License Plate (Single class detection)
* **Annotation Format:** YOLO format (.txt files with bounding box coordinates)
* **Train/Validation/Test Split Strategy:** The dataset originally provided training and validation directories. To ensure a rigorous and unbiased final evaluation, the original validation set (168 images) was programmatically split 50/50 using Python into dedicated validation and unseen test sets (84 images each).
* **Dataset Balance Analysis:** The dataset is checked for class and environmental balance (ensuring a diverse distribution of day/night lighting, front/side angles, and different plate shapes) to prevent model bias and ensure robust performance.

## 5. Data Preparation & Preprocessing
To prepare our dataset for YOLOv8 training, the following preprocessing steps and configurations are applied:

* **Image Dimensions & Resizing:** Images are resized to a standard dimension of $640 \times 640$ pixels, which balances computational speed and detection accuracy for YOLOv8.
* **Train/Validation/Test Split:** The dataset is split into:
  * **Training set:** 80% (used to train the model weights)
  * **Validation set:** 10% (used to tune hyperparameters and monitor performance during training)
  * **Test set:** 10% (kept unseen for final evaluation)
* **Data Augmentation:** Enabled standard YOLOv8 augmentations (such as random scaling, mosaic augmentation, and color space adjustments) to enhance model robustness against varying lighting and angles in Oman's street environments.
* **Label/Annotation Format:** Standard YOLO text format (`.txt`) where each image has a corresponding text file containing normalized bounding box coordinates: `[class_id, x_center, y_center, width, height]`.

In [1]:
# Install Ultralytics YOLOv8 library
%pip install ultralytics

# Verify the installation and setup
import ultralytics
from ultralytics import YOLO

ultralytics.checks()

Ultralytics 8.4.162  Python-3.14.7 torch-2.14.0+cpu CPU (13th Gen Intel Core i7-13620H)
Setup complete  (16 CPUs, 15.7 GB RAM, 264.7/474.7 GB disk)


In [2]:
import os
import shutil

# Paths for current validation directories
val_images_dir = "data/images/val"
val_labels_dir = "data/labels/val"

# Paths for new test directories
test_images_dir = "data/images/test"
test_labels_dir = "data/labels/test"

# Create test directories if they do not exist
os.makedirs(test_images_dir, exist_ok=True)
os.makedirs(test_labels_dir, exist_ok=True)

# Get all images in the validation directory
images = [f for f in os.listdir(val_images_dir) if f.endswith(('.jpg', '.jpeg', '.png'))]

# Split half of the validation images to be dedicated for the test folder
test_files = images[:len(images)//2]

for img_name in test_files:
    # Move the image file
    src_img = os.path.join(val_images_dir, img_name)
    dst_img = os.path.join(test_images_dir, img_name)
    shutil.move(src_img, dst_img)

    # Move the associated label file (.txt)
    base_name = os.path.splitext(img_name)[0]
    label_name = base_name + ".txt"
    
    src_label = os.path.join(val_labels_dir, label_name)
    dst_label = os.path.join(test_labels_dir, label_name)
    
    if os.path.exists(src_label):
        shutil.move(src_label, dst_label)

print(f"Successfully created the test folder and moved {len(test_files)} images and labels into it!")

Successfully created the test folder and moved 84 images and labels into it!


## 6. Model Training
We will use the YOLOv8 nano (`yolov8n.pt`) pre-trained model as a starting point (Transfer Learning). This allows the model to leverage pre-learned edge and shape features while training specifically on our license plate dataset.

In [3]:
from ultralytics import YOLO

# Load a pre-trained YOLOv8 nano model
model = YOLO('yolov8n.pt')

# Start training the model with our dataset configuration
results = model.train(
    data='data.yaml',    # Path to the dataset configuration file
    epochs=25,           # Number of training epochs
    imgsz=640,           # Image size for training
    batch=16             # Batch size
)

New https://pypi.org/project/ultralytics/8.4.163 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.162  Python-3.14.7 torch-2.14.0+cpu CPU (13th Gen Intel Core i7-13620H)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.9

## 7. Model Testing & Final Evaluation
To ensure an unbiased and rigorous evaluation of the trained model, we utilize the dedicated unseen **Test Set (`test`)**. This evaluates how well the model generalizes to completely new, real-world license plate images that were never used during training or hyperparameter tuning.

In [1]:
from ultralytics import YOLO

# 1. Load the best trained model weights
model = YOLO('runs/detect/train/weights/best.pt')

# 2. Evaluate performance on the test dataset split
metrics = model.val(data='data.yaml', split='test')

# 3. Run predictions on test images to visualize results
results = model.predict(
    source='data/images/test',  # Path to test images
    conf=0.25,                  # Confidence threshold
    save=True                   # Save images with detected bounding boxes
)

print("Final testing, evaluation, and predictions completed successfully!")

Ultralytics 8.4.162  Python-3.14.7 torch-2.14.0+cpu CPU (13th Gen Intel Core i7-13620H)
Model summary (fused): 72 layers, 3,005,843 parameters, 0 gradients, 8.1 GFLOPs
WARNING val: Slow image access detected (ping: 0.30.0 ms, read: 27.519.1 MB/s, size: 230.2 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning C:\Users\USER PC\OneDrive\Desktop\computer vision\computer vision full project\data\labels\test... 84 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 84/84 178.4it/s 0.5s0.2s
val: New cache created: C:\Users\USER PC\OneDrive\Desktop\computer vision\computer vision full project\data\labels\test.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.4it/s 4.2s1.0ss
                   all         84         84      0.999          1      0.995      0.893
Speed: 1.0ms preprocess, 32.5ms inference, 0.0ms loss, 0.6ms p

## 8. Error Analysis & Discussion
Although our YOLOv8 model achieved exceptional performance metrics on the unseen test set (Precision: $0.999$, Recall: $1.0$, mAP50: $0.995$), conducting an error analysis is crucial to understand potential edge cases and limitations:

1. **Illumination and Glare Issues:** 
   * *Observation:* Extreme sunlight reflections or night headlights on license plates can occasionally obscure characters.
   * *Potential Impact:* Might reduce the detection confidence score, even if the bounding box is successfully located.

2. **Extreme Angles and Occlusion:**
   * *Observation:* Side-view angles or partial obstructions (such as dirt or frame covers) alter the standard rectangular geometry of the plates.
   * *Mitigation:* Handled effectively by our data augmentation pipeline (scaling and mosaic), but remains a theoretical edge case.

3. **Background Confusion (False Positives):**
   * *Observation:* Similar rectangular structures on vehicle bodies (like front grills or bumper lines) could occasionally mimic plate shapes.
   * *Mitigation:* Minimized successfully by our high precision threshold, ensuring near-zero false alarms.

## 8. Build an Interactive Demo
To demonstrate that our trained YOLOv8 model works effectively outside the core training environment, we build a simple and interactive web application using **Gradio**. This allows any user to upload a custom car image and instantly visualize the detected license plates with bounding boxes and confidence scores.

In [5]:
%pip install gradio

Note: you may need to restart the kernel to use updated packages.


In [2]:
import gradio as gr
from ultralytics import YOLO
import cv2

# Load our trained YOLOv8 model weights
model = YOLO('runs/detect/train/weights/best.pt')

def detect_license_plate(image):
    # Convert Gradio RGB image to BGR expected by OpenCV/YOLO
    img_bgr = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    
    # Run prediction on the BGR image
    results = model.predict(source=img_bgr, conf=0.25)
    
    # Plot the bounding boxes on the image (Ultralytics plot returns BGR format)
    res_plotted = results[0].plot()
    
    # Convert BGR image to RGB for correct color display in Gradio
    res_rgb = cv2.cvtColor(res_plotted, cv2.COLOR_BGR2RGB)
    
    return res_rgb

# Create the Gradio web interface with the unified project title
demo = gr.Interface(
    fn=detect_license_plate,
    inputs=gr.Image(type="numpy", label="Upload Car Image"),
    outputs=gr.Image(type="numpy", label="Detected License Plate"),
    title="Automatic Number Plate Recognition (ANPR) System using YOLOv8",
    description="Upload a car image to test the trained YOLOv8 license plate detection model."
)

# Launch the demo locally
demo.launch(inline=False)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
